# 0825_lsw_012_psi_triggered_retrain

011에서 판정기준 변화(concept drift)는 type0/2/3에서 강하고 type4에서는 거의 없다는 걸
확인했다. 반대로 kimjaehak의 PSI(covariate shift, 입력 분포 변화) 분석에서는 **type4가
압도적으로 크고(13개 중 11개 피처 PSI≥0.25), type1도 일부 피처가 PSI≥1**로 나왔다 —
즉 이 두 유형은 concept drift보다 covariate shift가 더 지배적인 유형으로 보인다.

지금까지(006/008) 시도한 재학습은 **무조건적 재학습**(트리거 없이 매번 재학습)이었고, 그
결과가 나빴던 유형(type0/2/3)은 concept drift가 문제인 유형이었다 — 그 경우 재학습이 안
통하는 게 당연하다(새 기준을 보여주는 데이터 자체가 없으므로). 이번 노트북은 다른 질문을
던진다: **"covariate shift가 지배적인 유형(type1, type4)에서, PSI로 감지한 시점에만
재학습하면 도움이 되는가?"**

세 정책을 type1/type4에 한정해서 비교한다:

1. **frozen**: 최초(0~40%) 모델을 끝까지 재학습 없이 사용.
2. **PSI-triggered retrain**: 매 구간 시작 전, 그 구간 직전에 새로 확정된 데이터의 피처
   분포를 원래 Train 분포와 비교해 PSI를 계산한다. 유형의 피처들 중 **PSI 중앙값이 0.25
   (kimjaehak이 쓴 "심각" 기준)를 넘으면** 그 시점까지 확정된 모든 데이터로 재학습하고,
   넘지 않으면 이전 모델을 그대로 유지한다.
3. **always retrain(무조건 재학습)**: 트리거 없이 매 구간 직전 확정된 데이터로 항상
   재학습한다 — "PSI로 거르는 것 자체가 의미가 있는지" 비교하기 위한 대조군.

세 정책 모두 임계값은 매 구간 직전 확정된 데이터로 다시 고른다(008의 adaptive_threshold와
동일한 방식) — 그래야 "모델을 언제 재학습하느냐"만 순수하게 비교할 수 있다.


## 1. 설정과 라이브러리

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, confusion_matrix, roc_auc_score
from xgboost import XGBClassifier

EXPERIMENT_ID = "0825_lsw_012_psi_triggered_retrain"
RANDOM_STATE = 42
DATA_PATH = Path("../data/raw/dataset.csv")
TARGET = "class"
TIME_COLUMN = "timestamp"
RECORD_ID = "record_id"
TARGET_TYPES = [1, 4]  # covariate shift가 지배적인 두 유형만

COST_SCENARIOS = {"1:10": (1, 10), "1:100": (1, 100)}
PSI_TRIGGER_THRESHOLD = 0.25  # kimjaehak 0823_kimjaehak_002의 "심각" 기준과 동일

assert DATA_PATH.exists(), f"파일을 찾을 수 없습니다: {DATA_PATH.resolve()}"
print("experiment:", EXPERIMENT_ID)


experiment: 0825_lsw_012_psi_triggered_retrain


## 2. 데이터 로딩·전처리 (006/008과 동일한 cut point)

In [2]:
raw_df = pd.read_csv(DATA_PATH, low_memory=False)
source_index_column = raw_df.columns[0]
if source_index_column.startswith("Unnamed:"):
    raw_df = raw_df.rename(columns={source_index_column: RECORD_ID})
elif source_index_column != RECORD_ID:
    raise ValueError(f"예상하지 못한 첫 번째 컬럼: {source_index_column}")
assert raw_df[RECORD_ID].is_unique, "record_id가 고유하지 않습니다."

dedup_columns = [c for c in raw_df.columns if c not in {RECORD_ID, TIME_COLUMN}]
duplicate_mask = raw_df.duplicated(subset=dedup_columns, keep="first")
clean_df = raw_df.loc[~duplicate_mask].copy().reset_index(drop=True)

clean_df[TIME_COLUMN] = pd.to_datetime(clean_df[TIME_COLUMN], errors="raise", utc=True)
clean_df = clean_df.sort_values([TIME_COLUMN, RECORD_ID], kind="stable").reset_index(drop=True)

feature_columns_all = [c for c in clean_df.columns if c not in {RECORD_ID, TIME_COLUMN, TARGET}]
timestamps = clean_df[TIME_COLUMN]


def cut_at(fraction):
    sizes = timestamps.value_counts(sort=False).sort_index()
    cum = sizes.cumsum().to_numpy()
    idx = int(np.searchsorted(cum, len(clean_df) * fraction, side="left"))
    idx = min(idx, len(sizes) - 1)
    return sizes.index[idx]


cut_points = {f: cut_at(f) for f in [0.40, 0.60, 0.80]}
print("cut points:", cut_points)


cut points: {0.4: Timestamp('1970-08-25 14:06:02+0000', tz='UTC'), 0.6: Timestamp('1970-09-28 05:48:26+0000', tz='UTC'), 0.8: Timestamp('1970-10-13 13:14:26+0000', tz='UTC')}


## 3. 평가 함수 (003~011과 동일)

In [3]:
def slip_rate(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    actual_positive = y_true == 1
    if actual_positive.sum() == 0:
        return 0.0
    fn = ((y_pred == 0) & actual_positive).sum()
    return fn / actual_positive.sum()


def volume_reduction(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    actual_negative = y_true == 0
    if actual_negative.sum() == 0:
        return 0.0
    tn = ((y_pred == 0) & actual_negative).sum()
    return tn / actual_negative.sum()


def total_cost(y_true, y_pred, cost_fp, cost_fn):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    fn = ((y_pred == 0) & (y_true == 1)).sum()
    fp = ((y_pred == 1) & (y_true == 0)).sum()
    return fn * cost_fn + fp * cost_fp


def select_threshold(y_val, proba_val, max_slip_rate=0.01):
    candidates = np.sort(np.unique(proba_val))[::-1]
    for t in candidates:
        y_pred = (proba_val >= t).astype(int)
        if slip_rate(y_val, y_pred) <= max_slip_rate:
            return float(t)
    return 0.0


def evaluate_at_threshold(y_true, proba, threshold):
    y_pred = (proba >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    result = {
        "threshold": threshold,
        "n": len(y_true),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
        "slip_rate": slip_rate(y_true, y_pred),
        "volume_reduction": volume_reduction(y_true, y_pred),
        "pr_auc": average_precision_score(y_true, proba) if len(np.unique(y_true)) > 1 else float("nan"),
    }
    for name, (cost_fp, cost_fn) in COST_SCENARIOS.items():
        result[f"total_cost_{name}"] = total_cost(y_true, y_pred, cost_fp, cost_fn)
    return result


def get_non_constant_columns(candidate_columns, train_frame):
    nunique = train_frame[candidate_columns].nunique()
    return nunique[nunique > 1].index.tolist()


def build_model():
    return XGBClassifier(
        random_state=RANDOM_STATE,
        n_jobs=-1,
        tree_method="hist",
        objective="binary:logistic",
        eval_metric="logloss",
    )


## 4. PSI 계산 함수 (kimjaehak `0823_kimjaehak_002`와 동일한 방식)

기준 분포(reference)의 고유값이 10개 이하면 인접 고유값의 중점을 경계로, 그보다 많으면
기준 분포의 10분위 경계를 쓴다. 이 노트북에서는 **기준 분포 = 최초 Train(0~40%)**으로
고정하고, 비교 분포는 매 구간 직전에 새로 확정된 데이터로 바꿔가며 PSI를 다시 잰다.

In [4]:
def psi_bin_edges(reference_values):
    unique_vals = np.sort(np.unique(reference_values))
    if len(unique_vals) <= 10:
        if len(unique_vals) <= 1:
            return np.array([-np.inf, np.inf])
        midpoints = (unique_vals[:-1] + unique_vals[1:]) / 2
        return np.concatenate([[-np.inf], midpoints, [np.inf]])
    deciles = np.quantile(reference_values, np.linspace(0, 1, 11))
    edges = np.unique(deciles)
    edges[0], edges[-1] = -np.inf, np.inf
    if len(edges) < 2:
        return np.array([-np.inf, np.inf])
    return edges


def compute_psi(reference_values, comparison_values, epsilon=1e-4):
    edges = psi_bin_edges(reference_values)
    ref_counts, _ = np.histogram(reference_values, bins=edges)
    comp_counts, _ = np.histogram(comparison_values, bins=edges)
    ref_share = ref_counts / max(len(reference_values), 1) + epsilon
    comp_share = comp_counts / max(len(comparison_values), 1) + epsilon
    return float(np.sum((ref_share - comp_share) * np.log(ref_share / comp_share)))


def type_median_psi(feature_columns, reference_df, comparison_df):
    if len(comparison_df) == 0:
        return np.nan, {}
    per_feature = {
        col: compute_psi(reference_df[col].to_numpy(), comparison_df[col].to_numpy())
        for col in feature_columns
    }
    return float(np.median(list(per_feature.values()))), per_feature


## 5. 최초(0~40%) 모델 학습 — 세 정책의 공통 출발점

In [5]:
train_mask0 = timestamps <= cut_points[0.40]

initial_models = {}
initial_feature_columns = {}
initial_train_df = {}

for inspection_type in TARGET_TYPES:
    type_mask = clean_df["inspection_type"] == inspection_type
    train_df = clean_df.loc[train_mask0 & type_mask]
    feature_columns = get_non_constant_columns(feature_columns_all, train_df)

    model = build_model()
    model.fit(train_df[feature_columns], train_df[TARGET])

    initial_models[inspection_type] = model
    initial_feature_columns[inspection_type] = feature_columns
    initial_train_df[inspection_type] = train_df

    print(f"type {inspection_type}: train n={len(train_df)}, 유효 피처 {len(feature_columns)}개")


type 1: train n=26944, 유효 피처 34개
type 4: train n=2466, 유효 피처 24개


## 6. 세 정책 시뮬레이션 (구간별 PSI 계산 + 트리거 판단 + 평가)

- Val 구간 3개: (40~60%], (60~80%], (80~100%]
- **PSI 비교 대상**: 원래 Train(reference) vs **그 시점까지 누적으로 확정된 모든 데이터**
  — "지금까지 실제로 관측된 것 전체가 원래 학습 분포와 얼마나 달라졌는가"를 보는 방식.
- **재학습 시(트리거 발동 또는 always 정책)**: 원래 Train 시작부터 확정 시점까지
  **누적(expanding)** 데이터로 재학습한다.
- 1구간(40~60%)은 아직 확정된 사후 데이터가 없어 PSI를 잴 수 없으므로, 세 정책 모두
  최초 모델을 그대로 쓴다(자체 검증 포인트, 008과 동일한 관행).

In [6]:
val_bounds = [
    (cut_points[0.40], cut_points[0.60]),
    (cut_points[0.60], cut_points[0.80]),
    (cut_points[0.80], None),
]

results = []
psi_log = []

for inspection_type in TARGET_TYPES:
    type_mask = clean_df["inspection_type"] == inspection_type
    feature_columns = initial_feature_columns[inspection_type]
    reference_df = initial_train_df[inspection_type]

    active_model_psi = initial_models[inspection_type]
    active_model_always = initial_models[inspection_type]
    psi_cols = feature_columns
    always_cols = feature_columns

    for step, (val_lo, val_hi) in enumerate(val_bounds, start=1):
        val_mask = (timestamps > val_lo) & type_mask
        if val_hi is not None:
            val_mask &= timestamps <= val_hi
        val_df = clean_df.loc[val_mask]

        confirmed_hi = val_hi if step == 1 else val_lo
        confirmed_mask = (timestamps > cut_points[0.40]) & (timestamps <= confirmed_hi) & type_mask
        confirmed_df = clean_df.loc[confirmed_mask]

        if step == 1:
            # 1구간은 아직 학습 이후 확정된 사후 데이터가 없어 PSI를 잴 수 없다(008과 동일한 관행).
            median_psi, per_feature_psi = np.nan, {}
            triggered = False
        else:
            # PSI 비교 대상 = 원래 Train(reference) vs 그 시점까지 누적으로 확정된 모든 데이터.
            # "지금까지 실제로 관측된 것 전체가 원래 학습 분포와 얼마나 달라졌는가"를 보는 방식.
            median_psi, per_feature_psi = type_median_psi(feature_columns, reference_df, confirmed_df)
            triggered = bool(median_psi >= PSI_TRIGGER_THRESHOLD) if not np.isnan(median_psi) else False

        psi_log.append({
            "inspection_type": inspection_type, "step": step,
            "확정_누적_n": len(confirmed_df),
            "median_psi": median_psi, "trigger_fired": triggered,
        })

        # --- 재학습이 필요한 두 정책(psi_triggered, always) 공통: 확정 데이터 누적 재학습 ---
        def retrain(confirmed_df):
            cols = get_non_constant_columns(feature_columns, confirmed_df) if len(confirmed_df) > 0 else feature_columns
            cols = cols or feature_columns
            m = build_model()
            m.fit(confirmed_df[cols], confirmed_df[TARGET])
            return m, cols

        if step > 1:
            if triggered:
                active_model_psi, psi_cols = retrain(confirmed_df)
            active_model_always, always_cols = retrain(confirmed_df)

        for policy, model, cols in [
            ("frozen", initial_models[inspection_type], feature_columns),
            ("psi_triggered", active_model_psi, psi_cols),
            ("always_retrain", active_model_always, always_cols),
        ]:
            confirmed_proba = model.predict_proba(confirmed_df[cols])[:, 1]
            threshold = select_threshold(confirmed_df[TARGET], confirmed_proba)
            val_proba = model.predict_proba(val_df[cols])[:, 1]
            result = evaluate_at_threshold(val_df[TARGET], val_proba, threshold)
            result.update({"inspection_type": inspection_type, "step": step, "policy": policy})
            results.append(result)

results_df = pd.DataFrame(results).set_index(["inspection_type", "step", "policy"]).sort_index()
psi_log_df = pd.DataFrame(psi_log).set_index(["inspection_type", "step"])
print("완료")

완료


## 7. PSI 트리거 로그 — 언제 재학습이 발동됐는가

In [7]:
psi_log_df


확정_누적_n  median_psi  trigger_fired
inspection_type step                                    
1               1        7097         NaN          False
                2        7097    0.161360          False
                3       17591    0.152005          False
4               1        1052         NaN          False
                2        1052    0.672672           True
                3        2215    0.555012           True

## 8. 자체 검증 — 1구간은 세 정책이 모두 같아야 한다

In [8]:
check_rows = []
for inspection_type in TARGET_TYPES:
    row = {"inspection_type": inspection_type}
    for policy in ["frozen", "psi_triggered", "always_retrain"]:
        row[f"{policy}_threshold"] = results_df.loc[(inspection_type, 1, policy), "threshold"]
    check_rows.append(row)
pd.DataFrame(check_rows).set_index("inspection_type")


,frozen_threshold,psi_triggered_threshold,always_retrain_threshold
inspection_type,,,
1,0.000007,0.000007,0.000007
4,0.000110,0.000110,0.000110


## 9. 정책별 총비용 비교 (구간별)

In [9]:
summary_cols = ["threshold", "n", "tn", "fp", "fn", "tp", "slip_rate", "total_cost_1:10", "total_cost_1:100"]
results_df[summary_cols]


threshold      n     tn    fp   fn   tp  \
inspection_type step policy                                                    
1               1    always_retrain   0.000007   7097   1440  5399    2  256   
                     frozen           0.000007   7097   1440  5399    2  256   
                     psi_triggered    0.000007   7097   1440  5399    2  256   
                2    always_retrain   0.822695  10494  10169    86  116  123   
                     frozen           0.000007  10494   1275  8980    3  236   
                     psi_triggered    0.000007  10494   1275  8980    3  236   
                3    always_retrain   0.622190  11112  10293    35  616  168   
                     frozen           0.000006  11112   1167  9161    6  778   
                     psi_triggered    0.000006  11112   1167  9161    6  778   
4               1    always_retrain   0.000110   1052    800   247    0    5   
                     frozen           0.000110   1052    800   247    0    5   
                     psi_triggered    0.000110   1052    800   247    0    5   
                2    always_retrain   0.417285   1163   1102     1   60    0   
                     frozen           0.000110   1163    750   353   54    6   
                     psi_triggered    0.417285   1163   1102     1   60    0   
                3    always_retrain   0.751350    756    730     0   24    2   
                     frozen           0.000005    756      4   726    1   25   
                     psi_triggered    0.751350    756    730     0   24    2   

                                     slip_rate  total_cost_1:10  \
inspection_type step policy                                       
1               1    always_retrain   0.007752             5419   
                     frozen           0.007752             5419   
                     psi_triggered    0.007752             5419   
                2    always_retrain   0.485356             1246   
                     frozen           0.012552             9010   
                     psi_triggered    0.012552             9010   
                3    always_retrain   0.785714             6195   
                     frozen           0.007653             9221   
                     psi_triggered    0.007653             9221   
4               1    always_retrain   0.000000              247   
                     frozen           0.000000              247   
                     psi_triggered    0.000000              247   
                2    always_retrain   1.000000              601   
                     frozen           0.900000              893   
                     psi_triggered    1.000000              601   
                3    always_retrain   0.923077              240   
                     frozen           0.038462              736   
                     psi_triggered    0.923077              240   

                                     total_cost_1:100  
inspection_type step policy                            
1               1    always_retrain              5599  
                     frozen                      5599  
                     psi_triggered               5599  
                2    always_retrain             11686  
                     frozen                      9280  
                     psi_triggered               9280  
                3    always_retrain             61635  
                     frozen                      9761  
                     psi_triggered               9761  
4               1    always_retrain               247  
                     frozen                       247  
                     psi_triggered                247  
                2    always_retrain              6001  
                     frozen                      5753  
                     psi_triggered               6001  
                3    always_retrain              2400  
                     frozen                       826  
                     psi_trigge

## 10. 정책별 누적(3개 구간 합산) 총비용

In [10]:
agg = results_df.groupby(["inspection_type", "policy"])[["total_cost_1:10", "total_cost_1:100", "fn", "tn"]].sum()
agg


total_cost_1:10  total_cost_1:100   fn     tn
inspection_type policy                                                       
1               always_retrain            12860             78920  734  21902
                frozen                    23650             24640   11   3882
                psi_triggered             23650             24640   11   3882
4               always_retrain             1088              8648   84   2632
                frozen                     1876              6826   55   1554
                psi_triggered              1088              8648   84   2632

## 11. 결론 및 다음 단계

### 무슨 일이 일어났는지

- **type1**: median PSI가 2구간 0.161, 3구간 0.152로 **트리거 기준(0.25)을 한 번도 못 넘었다.**
  결과적으로 `psi_triggered` 정책은 3구간 내내 `frozen`과 완전히 동일했다.
- **type4**: median PSI가 2구간 0.673, 3구간 0.555로 **두 번 다 트리거가 발동했다.** 결과적으로
  `psi_triggered` 정책은 3구간 내내 `always_retrain`과 완전히 동일했다.

즉 이번 3-step 구조와 0.25라는 기준값에서는, PSI 트리거가 "가끔 켜지고 가끔 꺼지는" 중간 정책이
아니라 유형별로 **한쪽으로 완전히 쏠린 정책**이 됐다. 이 자체가 하나의 발견이다 — type1은
"PSI가 계속 기준 미만이라 사실상 계속 안전하다"는 뜻이고, type4는 "PSI가 계속 기준을 넘어
사실상 계속 위험 신호가 뜬다"는 뜻이다. kimjaehak의 최종(0~60%→80~100%) 비교에서 확인한
"type4는 압도적, type1은 경계선" 패턴과 방향이 일치한다.

### type1 — 재학습이 오히려 안전 기준을 크게 위반시켰다

`always_retrain`은 1:10 비용 기준으로는 더 싸 보인다(12,860 vs frozen 23,650). 하지만 이건
착시다. `always_retrain`의 2·3구간 Slip Rate는 각각 **48.5%, 78.6%**로, 목표(1% 이하)를
완전히 무너뜨린다 — 즉 실제 불량의 절반에서 대부분을 놓친다는 뜻이다. 이 프로젝트의 안전
기준으로 보면 `always_retrain`은 애초에 후보에서 제외돼야 하는 정책이고, `frozen`(=이번
설정에서는 `psi_triggered`와 동일)이 Slip Rate 0.77~1.26%로 목표에 근접하게 유지한다.
**PSI 트리거가 여기서는 "쓸데없는 재학습을 막아서" 결과적으로 옳은 선택을 했다.**

### type4 — 표본이 너무 작아 결론을 내리기 어렵다

PSI가 두 구간 다 트리거돼서 `psi_triggered`가 `always_retrain`과 같아졌는데, 구간별 결과가
극단적으로 요동친다(2구간 Slip Rate 90%→100%, 3구간 frozen 3.8% vs 재학습 92.3%). 원래
type4는 전체 데이터에서 불량이 99건뿐이고 이번 3-step의 각 구간에는 많아야 수십 건만 들어가서,
어느 정책을 써도 몇 건 차이로 결과가 뒤집힌다. **이 유형은 이번 실험으로도 "결론을 낼 수 없다"는
기존 판단(003~011에서 반복된 표본 부족 문제)이 그대로 재확인됐을 뿐이다.**

### 원래 질문에 대한 답

"covariate shift가 지배적인 유형에서는 재학습이 도움이 되는가?" — **부분적으로만 맞고,
더 중요한 사실이 하나 드러났다.** type1에서는 재학습(무조건이든 PSI 트리거든)이 도움은커녕
안전 기준을 크게 위반시켰다. type4는 표본 부족으로 판단 자체가 불가능하다. 즉 이번 실험은
"covariate shift니까 재학습하면 된다"는 가설을 뒷받침하지 못했다.

더 근본적인 발견은 이거다: **재학습 시점에 확정 데이터로 고른 임계값이, 다음 구간에서
안전 기준을 지키지 못하는 현상은 concept drift 유형(006/008)뿐 아니라 covariate shift
유형(type1)에서도 똑같이 나타난다.** 즉 "어떤 종류의 drift인가"와 무관하게, **이 데이터에서는
"지금까지 확정된 데이터로 고른 임계값이 다음 구간에서도 안전하다"는 가정 자체가 자주 깨진다는
것**이 더 상위의, 더 일관된 문제로 보인다.

### 다음 단계

- type1의 PSI 트리거 기준값(0.25)을 더 낮춰서(예: 0.15) 진짜로 "가끔 켜지는" 중간 정책을
  만들어 재검증해볼 수 있다 — 이번엔 기준값이 항상 미달이라 트리거의 변별력 자체를 테스트하지
  못했다.
- type4는 표본을 늘릴 방법이 없는 한(정적 스냅샷이라 불가능) 이 이상의 결론은 무리다.
- "확정 데이터 임계값이 다음 구간에서 자주 깨진다"는 상위 발견은 011의 "Test 구간과 급변
  구간이 겹친다"는 결론과 같은 방향이다 — 두 실험 모두 결국 "이 데이터에는 threshold
  일반화 자체가 어려운 근본적인 구조가 있다"는 하나의 결론으로 수렴한다.
